# Demo 02 — shuffle join vs broadcast join

Цель: сравнить два физических плана на одинаковых данных. Мы оцениваем не только wall-clock time, но и operators, `Exchange`, stages и shuffle metrics.

In [ ]:
from pyspark.sql import functions as F
from mentor_spark_lab.notebook_support import create_spark_session, explain_as_text
from mentor_spark_lab.pipeline import MarketplacePipeline

spark = create_spark_session("lesson04-notebook-join-experiment")
spark.sparkContext.setLogLevel("WARN")
pipeline = MarketplacePipeline(spark)
input_root = "/workspace/labs/spark/data/lesson04"

In [ ]:
purchases = pipeline.valid_purchases(pipeline.read_events(input_root))
customers = pipeline.read_customers(input_root)
print(f"input partitions: events={purchases.rdd.getNumPartitions()}, customers={customers.rdd.getNumPartitions()}")

## A. Запрещаем автоматический broadcast

Ожидание: обе стороны repartition по `customer_id`, затем Spark выполняет `SortMergeJoin`.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
shuffle_join = purchases.join(customers, "customer_id").groupBy("country").count()
shuffle_plan = explain_as_text(shuffle_join)
print(shuffle_plan)
assert "SortMergeJoin" in shuffle_plan
assert shuffle_plan.count("Exchange") >= 2

In [ ]:
shuffle_rows = shuffle_join.collect()
assert len(shuffle_rows) == 5
sorted(shuffle_rows, key=lambda row: row.country)

## B. Явный broadcast маленького dimension

Ожидание: shuffle большой fact-side для join исчезает, появляется `BroadcastExchange` и `BroadcastHashJoin`. Агрегация по стране всё ещё требует собственного `Exchange`.

In [ ]:
broadcast_join = purchases.join(F.broadcast(customers), "customer_id").groupBy("country").count()
broadcast_plan = explain_as_text(broadcast_join)
print(broadcast_plan)
assert "BroadcastHashJoin" in broadcast_plan
assert "BroadcastExchange" in broadcast_plan

In [ ]:
broadcast_rows = broadcast_join.collect()
assert sorted(shuffle_rows) == sorted(broadcast_rows)
print("PASS equal_business_result: shuffle and broadcast outputs match")

### Вопросы для deep dive

1. Почему `broadcast` не удалил все `Exchange`?
2. Как подтвердить, что dimension помещается в memory budget каждого executor?
3. Может ли AQE заменить запланированный join во время исполнения?
4. Почему сравнивать только секунды на одном ноутбуке недостаточно?

In [ ]:
spark.stop()
print("PASS shuffle_and_joins_demo")